# 03 — Silver → Gold

Calculamos las tablas `perfil_clientes` y `ventas_productos` con **DuckDB** a partir
de las compras de los dos primeros meses. Estos agregados describen el comportamiento
de los clientes y el desempeño de los productos antes de aplicar los incentivos.

**Criterios de análisis:**

- Definimos la ventana como los dos meses calendario consecutivos desde el primer mes de Silver.
- Usamos el último día del segundo mes como referencia para la recencia.
- Identificamos la franja preferida por número de compras; resolvemos empates en el orden mañana, almuerzo, tarde y noche.
- Identificamos la categoría preferida por unidades compradas, incluidos canjes; resolvemos empates por orden alfabético.
- Calculamos las unidades vendidas y los ingresos únicamente con líneas monetarias.
- Consideramos rezagado un producto cuando `indice_ventas < 0.5`.

Los perfiles alimentan la segmentación, mientras las ventas permiten identificar
los productos que necesitan un incentivo.


In [1]:
from pathlib import Path
import duckdb

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "doc.md").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Ejecutar desde la raíz del proyecto o notebooks/.")
SILVER = PROJECT_ROOT / "data" / "silver"
GOLD = PROJECT_ROOT / "data" / "gold"

MESES_VENTANA = 2
UMBRAL_REZAGO = 0.5
FRANJAS = ["mañana", "almuerzo", "tarde", "noche"]


def sql_text(value):
    return "'" + str(value).replace("'", "''") + "'"


connection = duckdb.connect(":memory:")
connection.execute("SET threads = 1")
for table in ("clientes", "productos", "restaurantes", "compras", "lineas_compra"):
    parquet = sql_text(SILVER / f"{table}.parquet")
    connection.execute(f"CREATE VIEW {table} AS SELECT * FROM read_parquet({parquet})")
print(f"DuckDB {duckdb.__version__}")

DuckDB 1.5.5


## Ventana de análisis

Tomamos los dos meses calendario consecutivos desde el primer mes de Silver.
Fijamos la fecha de referencia para la recencia en el último día del segundo mes,
justo antes del inicio de la aplicación de incentivos.


In [2]:
inicio_mes1, fin_ventana, fecha_referencia = connection.execute("""
    SELECT
        date_trunc('month', min(fecha_hora)) AS inicio_mes1,
        date_trunc('month', min(fecha_hora)) + INTERVAL 2 MONTH AS fin_ventana,
        date_trunc('month', min(fecha_hora)) + INTERVAL 2 MONTH - INTERVAL 1 DAY AS fecha_referencia
    FROM compras
""").fetchone()

connection.execute(f"""
    CREATE VIEW ventana AS
    SELECT * FROM compras
    WHERE fecha_hora >= {sql_text(inicio_mes1)}::TIMESTAMP
      AND fecha_hora < {sql_text(fin_ventana)}::TIMESTAMP
""")
print(f"Ventana: [{inicio_mes1.date()}, {fin_ventana.date()}) — fecha de referencia: {fecha_referencia.date()}")

Ventana: [2025-01-01, 2025-03-01) — fecha de referencia: 2025-02-28


## `perfil_clientes`

Resumimos el comportamiento de cada cliente en una fila: recencia, frecuencia mensual,
gasto promedio, franja horaria preferida y categoría preferida. Utilizamos estas variables
como base para identificar grupos con hábitos de compra similares.


In [3]:
franjas_sql = "[" + ", ".join(sql_text(franja) for franja in FRANJAS) + "]"

perfil_clientes = connection.execute(f"""
    WITH franja_compra AS (
        SELECT *,
            CASE
                WHEN extract(hour FROM fecha_hora) BETWEEN 6 AND 10 THEN 'mañana'
                WHEN extract(hour FROM fecha_hora) BETWEEN 11 AND 14 THEN 'almuerzo'
                WHEN extract(hour FROM fecha_hora) BETWEEN 15 AND 17 THEN 'tarde'
                ELSE 'noche'
            END AS franja
        FROM ventana
    ),
    conteo_franja AS (
        SELECT cliente_id, franja, count(*) AS n
        FROM franja_compra
        GROUP BY 1, 2
    ),
    franja_preferida AS (
        SELECT cliente_id, franja AS franja_preferida
        FROM (
            SELECT cliente_id, franja,
                   row_number() OVER (
                       PARTITION BY cliente_id
                       ORDER BY n DESC, list_position({franjas_sql}, franja)
                   ) AS posicion
            FROM conteo_franja
        )
        WHERE posicion = 1
    ),
    unidades_categoria_cliente AS (
        SELECT v.cliente_id, p.categoria, sum(l.cantidad) AS unidades
        FROM ventana v
        JOIN lineas_compra l ON l.compra_id = v.id
        JOIN productos p ON p.id = l.producto_id
        GROUP BY 1, 2
    ),
    categoria_preferida AS (
        SELECT cliente_id, categoria AS categoria_preferida
        FROM (
            SELECT cliente_id, categoria,
                   row_number() OVER (PARTITION BY cliente_id ORDER BY unidades DESC, categoria) AS posicion
            FROM unidades_categoria_cliente
        )
        WHERE posicion = 1
    ),
    resumen AS (
        SELECT cliente_id,
               count(*) AS num_compras,
               avg(monto_total) AS gasto_promedio,
               date_diff('day', max(fecha_hora)::DATE, {sql_text(fecha_referencia)}::DATE) AS recencia_dias
        FROM ventana
        GROUP BY 1
    )
    SELECT
        c.id AS cliente_id,
        r.recencia_dias,
        round(r.num_compras / {MESES_VENTANA}.0, 2) AS frecuencia_mensual,
        round(r.gasto_promedio, 2) AS gasto_promedio,
        fp.franja_preferida,
        cp.categoria_preferida
    FROM clientes c
    LEFT JOIN resumen r ON r.cliente_id = c.id
    LEFT JOIN franja_preferida fp ON fp.cliente_id = c.id
    LEFT JOIN categoria_preferida cp ON cp.cliente_id = c.id
    ORDER BY c.id
""").df()
perfil_clientes.head()

,cliente_id,recencia_dias,frecuencia_mensual,gasto_promedio,franja_preferida,categoria_preferida
0,1,17,1.0,23.75,almuerzo,acompañamientos
1,2,2,14.0,48.80,mañana,desayunos
2,3,1,18.5,99.88,noche,hamburguesas
3,4,2,2.0,21.98,almuerzo,acompañamientos
4,5,6,1.0,57.65,tarde,bebidas


## `ventas_productos`

Resumimos las unidades vendidas y los ingresos de cada producto y comparamos su desempeño
con el promedio de su categoría. Contamos únicamente líneas monetarias, ya que los canjes
representan uso de recompensas y no generan ingresos en la compra.


In [4]:
ventas_productos = connection.execute(f"""
    WITH lineas_dinero AS (
        SELECT l.producto_id, l.cantidad, l.subtotal
        FROM ventana v
        JOIN lineas_compra l ON l.compra_id = v.id
        WHERE NOT l.pagado_con_puntos
    ),
    ventas_producto AS (
        SELECT producto_id,
               sum(cantidad) AS unidades_vendidas,
               sum(subtotal) AS ingresos
        FROM lineas_dinero
        GROUP BY 1
    ),
    ventas_por_producto AS (
        SELECT p.id AS producto_id, p.categoria,
               coalesce(v.unidades_vendidas, 0)::BIGINT AS unidades_vendidas,
               coalesce(v.ingresos, 0) AS ingresos
        FROM productos p
        LEFT JOIN ventas_producto v ON v.producto_id = p.id
    ),
    promedio_por_categoria AS (
        SELECT categoria, avg(unidades_vendidas) AS promedio_categoria
        FROM ventas_por_producto
        GROUP BY 1
    )
    SELECT
        vp.producto_id,
        vp.categoria,
        vp.unidades_vendidas,
        round(vp.ingresos, 2) AS ingresos,
        round(pc.promedio_categoria, 4) AS promedio_categoria,
        round(vp.unidades_vendidas / pc.promedio_categoria, 4) AS indice_ventas,
        (vp.unidades_vendidas / pc.promedio_categoria) < {UMBRAL_REZAGO} AS es_rezagado
    FROM ventas_por_producto vp
    JOIN promedio_por_categoria pc ON pc.categoria = vp.categoria
    ORDER BY vp.producto_id
""").df()
ventas_productos.head()

,producto_id,categoria,unidades_vendidas,ingresos,promedio_categoria,indice_ventas,es_rezagado
0,1,hamburguesas,2959,106376.05,2235.0,1.3239,False
1,2,hamburguesas,2893,112537.70,2235.0,1.2944,False
2,3,hamburguesas,2946,53028.00,2235.0,1.3181,False
3,4,hamburguesas,142,4615.00,2235.0,0.0635,True
4,5,pollo,2472,79104.00,1823.5,1.3556,False


## Validación y exportación

Comprobamos que cada cliente y producto tenga una sola fila, que las variables requeridas
estén completas y que la clasificación de rezago coincida con el umbral establecido.
Guardamos los perfiles y las ventas en Parquet dentro de la capa Gold.


In [5]:
num_clientes = connection.execute("SELECT count(*) FROM clientes").fetchone()[0]
num_productos = connection.execute("SELECT count(*) FROM productos").fetchone()[0]

assert len(perfil_clientes) == num_clientes, "Cada cliente debe tener una fila en perfil_clientes"
assert perfil_clientes.isna().sum().sum() == 0, "perfil_clientes no debe tener nulos en la ventana analizada"
assert perfil_clientes["recencia_dias"].min() >= 0, "La recencia no puede ser negativa"
assert perfil_clientes["franja_preferida"].isin(FRANJAS).all(), "Franja fuera de las cuatro válidas"

assert len(ventas_productos) == num_productos, "Cada producto debe tener una fila en ventas_productos"
assert (ventas_productos["unidades_vendidas"] >= 0).all(), "Las unidades vendidas no pueden ser negativas"
assert ventas_productos["indice_ventas"].notna().all(), "indice_ventas no debe ser nulo"
assert (ventas_productos["es_rezagado"] == (ventas_productos["indice_ventas"] < UMBRAL_REZAGO)).all(), "es_rezagado no coincide con el umbral"

GOLD.mkdir(parents=True, exist_ok=True)
connection.register("perfil_clientes_df", perfil_clientes)
connection.register("ventas_productos_df", ventas_productos)
connection.execute(f"COPY (SELECT * FROM perfil_clientes_df ORDER BY cliente_id) TO {sql_text(GOLD / 'perfil_clientes.parquet')} (FORMAT PARQUET, COMPRESSION ZSTD)")
connection.execute(f"COPY (SELECT * FROM ventas_productos_df ORDER BY producto_id) TO {sql_text(GOLD / 'ventas_productos.parquet')} (FORMAT PARQUET, COMPRESSION ZSTD)")

for name, esperado in (("perfil_clientes", num_clientes), ("ventas_productos", num_productos)):
    filas = connection.execute(f"SELECT count(*) FROM read_parquet({sql_text(GOLD / f'{name}.parquet')})").fetchone()[0]
    assert filas == esperado, f"{name}: se exportaron {filas} filas, se esperaban {esperado}"
    print(f"{name}: {filas} filas exportadas a Gold.")

rezagados = int(ventas_productos["es_rezagado"].sum())
print(f"Productos rezagados (indice_ventas < {UMBRAL_REZAGO}): {rezagados} de {num_productos}.")
print("Distribución de franja_preferida:")
print(perfil_clientes["franja_preferida"].value_counts())

connection.close()

perfil_clientes: 1000 filas exportadas a Gold.
ventas_productos: 28 filas exportadas a Gold.
Productos rezagados (indice_ventas < 0.5): 4 de 28.
Distribución de franja_preferida:
franja_preferida
mañana      379
almuerzo    299
noche       166
tarde       156
Name: count, dtype: int64


## Resultados

Obtenemos un perfil por cliente y un resumen de ventas por producto. Estos agregados permiten
segmentar el comportamiento e identificar productos con baja rotación relativa.

Todos los clientes del escenario presentan compras en la ventana de análisis y todas
las categorías registran ventas. Estas condiciones permiten calcular los indicadores
sin imputar valores para clientes sin historial o categorías sin compras.
